# Data Cleaning 02 -- FRED Monthly Macro

## Input
`Data/Data_Collection/Initial/02_FRED/fred_monthly_macro.parquet`

## Purpose
Cleans monthly US macroeconomic data from the FRED API. Keyed on `date` only (no PERMNO). Key concerns addressed: series pulled with first-release vintage (point-in-time safe), derived factors (`_mom`, `_yoy`) inheriting NaN from their inputs and lookback windows, the `ism_prices`/`ppi_final` duplicate FRED ID bug (both PPIFIS), quarterly series stuffed into a monthly table (`m2_velocity`), and series that start late or are discontinued before the end of the sample.

## Stage 0: Load & Inspect
Basic shape, date range, column inventory, and dtype verification.

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts sorted descending, with first/last valid dates and flags for columns above 30% or 50% NaN
- Per-row NaN distribution and identification of the worst rows

## Stage 2: Specific Issue Checks
- **Date frequency check:** confirms monthly cadence, reports gap statistics
- **Duplicate FRED ID check:** confirms `ism_prices` and `ppi_final` are identical (both PPIFIS, correlation = 1.000000)
- **Quarterly series check:** `m2_velocity` identified as quarterly (88 obs, avg gap 91 days) in a monthly table
- **Derived factor leading NaN:** verifies that `_mom` factors have ~1 leading NaN, `_yoy` factors have ~12, and other derived columns (`copper_gold_ratio`, `trade_balance_12m_avg`) have expected patterns
- **Discontinued series:** lists all series whose last valid date falls before 2024-06-01
- **Value range sanity checks:** verifies key series fall within expected ranges (e.g., unemployment rate 0--20%, CPI-U 100--400, Michigan sentiment 40--120)

## Stage 4: Clean & Save

### Columns Dropped (12)
- `existing_home_sales`, `existing_home_sales_mom` -- 100% NaN, the FRED series `EXHOSLUSM495S` failed to pull entirely
- `m2_velocity` -- quarterly series (avg gap 91 days, 88 obs) stuffed into a monthly table, unusable at monthly frequency
- `ppi_final`, `ppi_final_mom`, `ppi_final_yoy` -- PPI Final Demand (FRED ID `PPIFIS`) only starts November 2009 when the BLS restructured PPI methodology, 31% NaN, missing the first 6 years of training data
- `ppi_core`, `ppi_core_mom` -- same issue, starts April 2010, 33% NaN
- `ism_prices` -- confirmed identical to `ppi_final` (both map to FRED ID `PPIFIS`), a bug in the collection code
- `lei` -- Conference Board Leading Economic Index, discontinued February 2020, entirely missing for the 2020--2024 test period
- `conf_board` -- OECD consumer confidence proxy (`CSCICP03USM665S`), last valid January 2024, missing the final 11 months, redundant with `umich_sentiment` which has full coverage

### Date Range Trimmed to 2004-01-01
The raw file starts 2003-01-01 to provide the 12-month lookback needed for `_yoy` and `trade_balance_12m_avg` computation during collection. Trimming to 2004 eliminates 12 leading NaN from all 6 `_yoy` factors, 11 leading NaN from `trade_balance_12m_avg`, 1 leading NaN from all 27 `_mom` factors, and 12 rows of 2003 data no longer needed.

### Forward-Fill (Limit 3 Months)
Applied after trimming to handle isolated reporting gaps. At monthly frequency a 3-month limit prevents filling across genuine data discontinuities while covering normal 1--2 month publication delays.

### Remaining Structural NaN (Kept, Not Imputed)
- `avg_hourly_earnings` -- BLS series `CES0500000003` starts March 2006, ~26 leading NaN (Jan 2004 -- Feb 2006). Same treatment as FRED daily `twexb`/`twexm` -- resolves when the merged dataset starts from 2006.

### Publication Lag Not Enforced Here
First-release macro data dated January is typically published in February and revised in March. The 2-month lag (factor dated January only enters the model from March onwards) is applied in the merge pipeline, not here. The dates in this file are observation dates, not availability dates.

## Output
`Data/Data_Collection/Cleaned/02_FRED/fred_monthly_macro_clean.parquet` -- 83 factor columns (down from 95 before cleaning)

In [1]:
# %% [markdown]
# # Data Cleaning: fred_monthly_macro.parquet
#
# Source: Data/Data_Collection/Initial/02_FRED/fred_monthly_macro.parquet
# Output: Data/Data_Collection/Cleaned/02_FRED/fred_monthly_macro_clean.parquet
#
# Monthly US macroeconomic data from the FRED API. No PERMNO — keyed on date only.
# Key concerns:
#   - Many series were pulled with first-release vintage (point-in-time safe)
#   - Derived factors (_mom, _yoy) inherit NaN from their inputs and from
#     the lookback window (first 1 or 12 months have no MoM/YoY)
#   - The ism_prices / ppi_final duplicate bug (both PPIFIS)
#   - Some series may be quarterly stuffed into monthly (m2_velocity)
#   - copper_gold_ratio was computed from daily gold averaged to monthly
#   - Publication lag (2-month) is NOT enforced here — that's for the merge pipeline

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/02_FRED/fred_monthly_macro.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/02_FRED')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — fred_monthly_macro")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique dates: {df['date'].nunique():,}")

factor_cols = [c for c in df.columns if c != 'date']

print(f"\nFactor columns ({len(factor_cols)}):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<40s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (5 rows, first 10 factors) ---")
print(df[['date'] + factor_cols[:10]].head(5).to_string(index=False))

print(f"\n--- Tail (5 rows, first 10 factors) ---")
print(df[['date'] + factor_cols[:10]].tail(5).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT — fred_monthly_macro")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN (sorted descending) ───────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN Summary ---")
print(f"  Factors with   0% NaN: {(col_nan_pct == 0).sum()}")
print(f"  Factors with  <5% NaN: {((col_nan_pct > 0) & (col_nan_pct < 5)).sum()}")
print(f"  Factors with 5-20% NaN: {((col_nan_pct >= 5) & (col_nan_pct < 20)).sum()}")
print(f"  Factors with 20-50% NaN: {((col_nan_pct >= 20) & (col_nan_pct < 50)).sum()}")
print(f"  Factors with ≥50% NaN: {(col_nan_pct >= 50).sum()}")

print(f"\n  {'Column':<40s} {'NaN %':>8s}  {'Count':>6s}  {'First Valid':>12s}  {'Last Valid':>12s}")
print("  " + "-" * 90)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    if count == 0:
        first = df['date'].min().date()
        last = df['date'].max().date()
    else:
        valid = df[df[col].notna()]['date']
        first = valid.min().date() if len(valid) > 0 else 'N/A'
        last = valid.max().date() if len(valid) > 0 else 'N/A'
    flag = ""
    if pct >= 50:
        flag = " ← DROP"
    elif pct >= 30:
        flag = " ← INVESTIGATE"
    elif pct > 0:
        flag = ""
    print(f"  {col:<40s} {pct:>7.2f}%  {count:>6d}  {str(first):>12s}  {str(last):>12s}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN:    {(row_nan == 0).sum():>5,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-5 NaN:  {((row_nan >= 1) & (row_nan <= 5)).sum():>5,d}")
print(f"  Rows with 6-20 NaN: {((row_nan > 5) & (row_nan <= 20)).sum():>5,d}")
print(f"  Rows with >20 NaN:  {(row_nan > 20).sum():>5,d}")
print(f"  Max NaN in any row: {row_nan.max()} out of {len(factor_cols)}")

# Show worst rows
worst_rows = df.loc[row_nan.nlargest(5).index, ['date']].copy()
worst_rows['n_nan'] = row_nan.nlargest(5).values
print(f"\n  5 rows with most NaN:")
print(worst_rows.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: SPECIFIC ISSUE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: SPECIFIC ISSUE CHECKS")
print("=" * 90)

# ── 2a. Date frequency check ────────────────────────────────────────────────
print(f"\n--- Date frequency ---")
date_diffs = df['date'].diff().dt.days.dropna()
print(f"  Most common gap: {date_diffs.mode().iloc[0]:.0f} days")
print(f"  Min gap: {date_diffs.min():.0f} days")
print(f"  Max gap: {date_diffs.max():.0f} days")
print(f"  Rows per year: {df.groupby(df['date'].dt.year).size().describe().to_string()}")

# ── 2b. Duplicate FRED ID check: ism_prices vs ppi_final ────────────────────
print(f"\n--- Check: ism_prices vs ppi_final duplication ---")
if 'ism_prices' in df.columns and 'ppi_final' in df.columns:
    both_valid = df[['ism_prices', 'ppi_final']].dropna()
    if len(both_valid) > 0:
        are_equal = (both_valid['ism_prices'] == both_valid['ppi_final']).all()
        corr = both_valid['ism_prices'].corr(both_valid['ppi_final'])
        print(f"  Rows where both are valid: {len(both_valid)}")
        print(f"  Identical values: {are_equal}")
        print(f"  Correlation: {corr:.6f}")
        if are_equal or corr > 0.999:
            print(f"  → CONFIRMED DUPLICATE. Drop ism_prices.")
    else:
        print(f"  No overlapping valid observations.")
else:
    for c in ['ism_prices', 'ppi_final']:
        if c not in df.columns:
            print(f"  {c} not found in columns")

# ── 2c. Quarterly series in monthly table ────────────────────────────────────
print(f"\n--- Check: Quarterly series in monthly table ---")
quarterly_suspects = ['m2_velocity']
for col in quarterly_suspects:
    if col not in df.columns:
        print(f"  {col}: not found")
        continue
    valid = df[df[col].notna()]['date']
    if len(valid) > 0:
        gaps = valid.diff().dt.days.dropna()
        avg_gap = gaps.mean()
        print(f"  {col}: {len(valid)} valid obs, avg gap = {avg_gap:.0f} days "
              f"({'quarterly' if avg_gap > 60 else 'monthly'})")

# ── 2d. Derived factor NaN pattern ──────────────────────────────────────────
# _mom factors need 1 prior month, _yoy factors need 12 prior months
print(f"\n--- Check: Derived factor leading NaN ---")
mom_cols = [c for c in factor_cols if c.endswith('_mom')]
yoy_cols = [c for c in factor_cols if c.endswith('_yoy')]
other_derived = ['copper_gold_ratio', 'trade_balance_12m_avg']

print(f"  MoM factors ({len(mom_cols)}): expect ~1 leading NaN each")
if mom_cols:
    mom_nan = df[mom_cols].isna().sum()
    print(f"    NaN range: {mom_nan.min()} to {mom_nan.max()}")

print(f"  YoY factors ({len(yoy_cols)}): expect ~12 leading NaN each")
if yoy_cols:
    yoy_nan = df[yoy_cols].isna().sum()
    print(f"    NaN range: {yoy_nan.min()} to {yoy_nan.max()}")

for col in other_derived:
    if col in df.columns:
        n = df[col].isna().sum()
        print(f"  {col}: {n} NaN")

# ── 2e. Series that end early (discontinued) ────────────────────────────────
print(f"\n--- Discontinued series (last valid date before 2024-06-01) ---")
discontinued = []
for col in factor_cols:
    valid = df[df[col].notna()]['date']
    if len(valid) > 0:
        last = valid.max()
        if last < pd.Timestamp('2024-06-01'):
            discontinued.append((col, last.date(), len(valid)))

if discontinued:
    for col, last_date, n_obs in sorted(discontinued, key=lambda x: x[1]):
        print(f"  {col:<40s} last valid: {last_date}  ({n_obs} obs)")
else:
    print(f"  None found")

# ── 2f. Value range sanity checks ───────────────────────────────────────────
print(f"\n--- Value range sanity checks ---")
checks = {
    'unrate':              (0, 20,      'Unemployment rate %'),
    'cpi_urban':           (100, 400,   'CPI-U index level'),
    'nonfarm_payrolls':    (100000, 200000, 'NFP in thousands'),
    'personal_income':     (5000, 30000, 'Personal income $B SAAR'),
    'umich_sentiment':     (40, 120,    'Michigan sentiment index'),
    'saving_rate':         (-5, 40,     'Personal saving rate %'),
    'housing_starts':      (200, 3000,  'Housing starts thousands SAAR'),
}

for col, (low, high, desc) in checks.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    n_out = ((vals < low) | (vals > high)).sum()
    if n_out > 0:
        print(f"  ⚠ {col:<30s} {n_out:>3d} values outside [{low}, {high}] — {desc}")
    else:
        print(f"  ✓ {col:<30s} all in [{low}, {high}]")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: SUMMARY — DECISIONS NEEDED BEFORE CLEANING")
print("=" * 90)

print(f"""
Review the output above and decide:

1. COLUMNS TO DROP:
   - ism_prices (if confirmed duplicate of ppi_final)
   - m2_velocity (if quarterly — too sparse for monthly panel)
   - Any series with ≥30% NaN
   - Any discontinued series missing the test period

2. DATE RANGE:
   - Raw file starts 2003-01-01 (to allow 12-month YoY computation)
   - Should we trim to 2004-01-01 now, or keep 2003 for context?
   - The _yoy factors will have NaN for 2003 regardless

3. FORWARD-FILL:
   - Monthly data: ffill limit=3 months for small gaps
   - Longer gaps: leave as NaN (structural)

4. DERIVED FACTOR NaN:
   - _mom leading NaN (1 month): filled by ffill
   - _yoy leading NaN (12 months): only affects 2003, trimmed if we cut to 2004
   - copper_gold_ratio, trade_balance_12m_avg: depends on input availability

5. PUBLICATION LAG:
   - NOT enforced here. The 2-month lag for first-release data
     must be applied in the merge pipeline.

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD & INSPECT — fred_monthly_macro

Shape: 264 rows × 96 columns
Date range: 2003-01-01 → 2024-12-01
Unique dates: 264

Factor columns (95):
    1. nonfarm_payrolls                         float64        
    2. unrate                                   float64        
    3. u6_rate                                  float64        
    4. participation                            float64        
    5. avg_hourly_earnings                      float64        
    6. avg_weekly_hours                         float64        
    7. jolts_openings                           float64        
    8. jolts_quits                              float64        
    9. cpi_urban                                float64        
   10. cpi_core                                 float64        
   11. cpi_food                                 float64        
   12. cpi_energy                               float64        
   13. cpi_shelter                              float64        
   14. cpi_servic

In [2]:
# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Cleaning decisions and rationale:**
#
# **Columns dropped (12):**
# - `existing_home_sales`, `existing_home_sales_mom` — 100% NaN. Failed to pull from FRED entirely.
# - `m2_velocity` — quarterly data in a monthly table (88 obs, avg gap 91 days). Unusable.
# - `ppi_final`, `ppi_final_mom`, `ppi_final_yoy` — only start Nov 2009 (31% NaN). The FRED ID
#   for PPI Final Demand (PPIFIS) was introduced in 2009 when the BLS restructured PPI.
# - `ppi_core`, `ppi_core_mom` — same issue, start Apr/May 2010 (33% NaN).
# - `ism_prices` — confirmed identical to `ppi_final` (same FRED ID PPIFIS). Duplicate.
# - `lei` — Conference Board LEI discontinued Feb 2020. Entirely missing for the test period.
# - `conf_board` — OECD consumer confidence proxy ends Jan 2024. Missing last 11 months.
#   Redundant with `umich_sentiment` which has full coverage.
#
# **Date range trimmed to 2004-01-01:**
# The raw file starts 2003-01-01 to allow 12-month YoY and 12-month rolling average
# computation. Trimming to 2004 eliminates all leading NaN from `_yoy` factors (12 months)
# and `trade_balance_12m_avg` (11 months). The 2003 data served its purpose during
# the collection phase and is no longer needed.
#
# **Remaining structural NaN (kept, not imputed):**
# - `avg_hourly_earnings` — starts March 2006 (~26 leading NaN after trim to 2004).
#   Same treatment as FRED daily `twexb`/`twexm` — will resolve when merged data starts from 2006.
#
# **Forward-fill (limit=3 months):**
# Applied after trimming to handle any isolated 1-month gaps from reporting delays.
#
# **Publication lag (2-month) NOT enforced here** — applied in the merge pipeline.
#
# **Factors retained: 83** (was 95 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

# ── 4a. Drop columns ────────────────────────────────────────────────────────
drop_cols = [
    # 100% NaN — failed to pull
    'existing_home_sales', 'existing_home_sales_mom',
    # Quarterly in monthly table
    'm2_velocity',
    # ≥30% NaN — late-starting PPI series (PPIFIS introduced 2009)
    'ppi_final', 'ppi_final_mom', 'ppi_final_yoy',
    'ppi_core', 'ppi_core_mom',
    # Confirmed duplicate of ppi_final (same FRED ID PPIFIS)
    'ism_prices',
    # Discontinued — missing test period
    'lei',
    # Discontinued — missing tail end, redundant with umich_sentiment
    'conf_board',
]

drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)

factor_cols = [c for c in df.columns if c != 'date']
print(f"\n  Dropped {len(drop_cols_present)} columns")
print(f"  Remaining: {len(factor_cols)} factor columns")

# ── 4b. Trim to 2004-01-01 ──────────────────────────────────────────────────
n_before = len(df)
df = df[df['date'] >= '2004-01-01'].reset_index(drop=True)
n_trimmed = n_before - len(df)
print(f"  Trimmed {n_trimmed} rows before 2004-01-01 ({n_before} → {len(df)})")

# ── 4c. Forward-fill remaining gaps (limit=3 months) ────────────────────────
nan_before = df[factor_cols].isna().sum().sum()
df = df.sort_values('date')
df[factor_cols] = df[factor_cols].ffill(limit=3)
nan_after = df[factor_cols].isna().sum().sum()
print(f"\n  Forward-fill (limit=3): {nan_before:,} → {nan_after:,} NaN")

# ── 4d. Final NaN report ────────────────────────────────────────────────────
nan_final = df[factor_cols].isna().sum()
nan_cols = nan_final[nan_final > 0]

if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN — all clean")
else:
    print(f"\n  Remaining NaN ({len(nan_cols)} columns):")
    for col, n in nan_cols.sort_values(ascending=False).items():
        first_valid = df[df[col].notna()]['date'].min().date()
        print(f"    {col:<35s} {n:>4d} NaN  (first valid: {first_valid})")

# ── 4e. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n} NaN)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<40s}{nan_str}")

print(f"\n  Sample (first 3 rows):")
print(df[['date'] + factor_cols[:8]].head(3).to_string(index=False))
print(f"\n  Sample (last 3 rows):")
print(df[['date'] + factor_cols[:8]].tail(3).to_string(index=False))

# ── 4f. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'fred_monthly_macro_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 4: CLEAN & SAVE

  Dropped 11 columns
  Remaining: 84 factor columns
  Trimmed 12 rows before 2004-01-01 (264 → 252)

  Forward-fill (limit=3): 26 → 26 NaN

  Remaining NaN (1 columns):
    avg_hourly_earnings                   26 NaN  (first valid: 2006-03-01)

  Final shape: 252 rows × 85 columns
  Factor columns: 84
  Date range: 2004-01-01 → 2024-12-01

  Factor list (84 columns):
      1. nonfarm_payrolls                        
      2. unrate                                  
      3. u6_rate                                 
      4. participation                           
      5. avg_hourly_earnings                       (26 NaN)
      6. avg_weekly_hours                        
      7. jolts_openings                          
      8. jolts_quits                             
      9. cpi_urban                               
     10. cpi_core                                
     11. cpi_food                                
     12. cpi_energy                           